In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("init_load_flag", "0")

init_load_flag = int(dbutils.widgets.get("init_load_flag"))


### **Data Reading From Silver**

In [0]:
df = spark.read.table("databricksete_cat.silver.customers_silver")

In [0]:
#Removing duplicates
df = df.dropDuplicates(subset= ["customer_id"])


In [0]:
## Filtering New vs Old records 

if init_load_flag == 0:
    df_old = spark.sql("select DimCustomerKey, customer_id, create_date, update_date from databricksete_cat.gold.DimCustomers")
    
else:
    df_old = spark.sql("select  0 DimCustomerKey, 0 customer_id, 0 create_date,  0 update_date from databricksete_cat.silver.customers_silver where 1=0")

In [0]:
#Renaming columns of df_old
df_old = df_old.withColumnRenamed("DimCustomerKey", "DimCustomerKey_old")\
                .withColumnRenamed("customer_id", "customer_id_old")\
                .withColumnRenamed("create_date", "create_date_old")\
                .withColumnRenamed("update_date", "update_date_old")    

In [0]:
#Applying join with the old records 

df_join = df.join(df_old, df.customer_id == df_old.customer_id_old, how = "left")


In [0]:
display(df_join)

In [0]:
#Separating new vs old records

df_new = df_join.filter(df_join.DimCustomerKey_old.isNull())
display(df_new)

df_old = df_join.filter(df_join.DimCustomerKey_old.isNotNull())
display(df_old)

In [0]:
# Preparing df_old

# Rebuild df_old from the joined data so this cell doesn't depend on stale state
df_old = df_join.filter(df_join.DimCustomerKey_old.isNotNull())

# 1. Drop unused columns
df_old = df_old.drop("customer_id_old", "update_date_old")

#2. Rename DimCustomerKey
df_old = df_old.withColumnRenamed("DimCustomerKey_old", "DimCustomerKey")

# 3. Rename create_date_old to create_date AND cast to timestamp using withColumn
df_old = df_old.withColumnRenamed("create_date_old", "create_date") \
               .withColumn("create_date", to_timestamp(col("create_date")))

# 4. Add update_date with the current timestamp
df_old = df_old.withColumn("update_date", current_timestamp())

df_old.printSchema()

In [0]:
#Preparing df_new 

#Dropping all the columns which are not required
df_new = df_new.drop("customer_id_old", "DimCustomerKey_old","update_date_old","create_date_old")


#Recreating "update_date" and "create_date" column with current timestamp
df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("create_date", current_timestamp())

In [0]:
display(df_new)

In [0]:
#Assining surrogate key from 1 
df_new = df_new.withColumn("DimCustomerKey", monotonically_increasing_id()+lit(1))



### **Adding max surrogate key**

In [0]:
if init_load_flag == 1:
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("select max(DimCustomerKey) as max_surrogate_key from databricksete_cat.gold.DimCustomers")
    #Converting df_maxsur to max_surrogate_key variable
    max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']

In [0]:
df_new = df_new.withColumn("DimCustomerKey", lit(max_surrogate_key)+col("DimCustomerKey"))

In [0]:
#Union of df_old and df_new

df_final = df_new.unionByName(df_old)

In [0]:
display(df_final)

In [0]:
from delta.tables import DeltaTable

In [0]:
#SCD Type 1

if init_load_flag == 0:
    
    dlt_obj = DeltaTable.forPath(spark, "abfss://gold@tjdatabricksete.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("trg").merge(df_final.alias("src"), "trg.DimCustomerKey = src.DimCustomerKey")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()
            
    
else:
    df_final.write.format("delta").mode("overwrite")\
        .saveAsTable("databricksete_cat.gold.DimCustomers")

In [0]:
%sql
--select * from databricksete_cat.gold.dimcustomers